In [1]:
import sys, os

import pickle

# import pandas as pd
# import some common libraries
import numpy as np
import json, cv2, random

import matplotlib.pyplot as plt

from PIL import Image
from PIL import ImageDraw
from vistats import annotate_brackets
# import seaborn as sns
import itertools
import traceback
import shutil
import scipy
import statistics as stat

from skimage.measure import label, regionprops
from scipy.spatial.distance import cdist
from scipy.ndimage import distance_transform_edt
from collections import defaultdict


from detectron2.structures import Instances, Boxes, BitMasks
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog, DatasetCatalog
from detectron2.data.datasets import register_coco_instances
from detectron2.engine import DefaultTrainer
import distutils.core
import torch, detectron2
from detectron2.utils.logger import setup_logger
# setup_logger()


from detectron2.data import build_detection_test_loader, MetadataCatalog
from detectron2.evaluation import COCOEvaluator, inference_on_dataset

In [2]:
rdir = "/home/gpu_data/data8/kinoshita_cfos/"
wsl_pyf = "/home/gpu_data/data8/kinoshita_cfos/EM_analysis/" 

# rdir = "/mnt/gpu_data/SSD3/"
# wsl_pyf = "/mnt/gpu_data/SSD3/EM_analysis/" 

In [3]:

def mask_iou(mask1, mask2):
    """
    2 つのバイナリマスク間の IoU（Intersection over Union; 重なり率）を計算する。
    mask1, mask2 : bool 型の 2 次元配列
    """
    intersection = np.logical_and(mask1, mask2).sum()
    union = np.logical_or(mask1, mask2).sum()
    return (intersection / np.sum(mask1)) if union > 0 else 0

def nms_mask_group(indices, pred_masks, scores, threshold=0.2):
    """
    与えられたインデックス群（同一クラス内のインデックス）について、
    スコアが高い順に並べ、マスク同士の IoU が threshold 以上のものを除外（代表１体のみ保持）します。
    
    Parameters:
      indices    : 同一クラス内の候補インスタンスの元のインデックス（numpy.array あるいは list）
      pred_masks : 全インスタンスのマスク配列（shape: (N, H, W)）
      scores     : 全インスタンスのスコア配列（1次元）
      threshold  : IoU のしきい値（例: 0.2）
      
    Returns:
      kept_indices : このグループ内で NMS 処理後に残るインスタンスの元のインデックスのリスト
    """
    if len(indices) == 0:
        return []
    # 候補をスコアの降順でソート（各候補は元のインデックス）
    sorted_indices = sorted(indices, key=lambda i: scores[i], reverse=True)
    kept_indices = []
    
    while sorted_indices:
        curr = sorted_indices.pop(0)
        kept_indices.append(curr)
        remain = []
        for idx in sorted_indices:
            iou = mask_iou(pred_masks[curr], pred_masks[idx])
            # print(iou)
            if iou < threshold:
                remain.append(idx)
            # IoUが threshold 以上なら、重複とみなして除外（代表でない側）
        sorted_indices = remain
    return kept_indices
    

def compute_centroid(mask):
    """
    2次元バイナリマスク画像から最初の連結領域の重心 (row, col) を計算する関数
    """
    labeled_mask = label(mask)
    regions = regionprops(labeled_mask)
    if not regions:
        raise ValueError("マスク画像に有効な領域が見つかりません。")
    return regions[0].centroid  # (row, col)

def extract_vec_indices_by_synapse(synapse_mask, vec_masks):
    """
    vec_masks における各マスク画像の重心座標を計算し、
    その座標が synapse_mask 内 (値が 0 以外) にある場合、その vec マスクのインデックスを返す。

    Parameters:
      synapse_mask : 2次元バイナリマスク (numpy.array)
      vec_masks    : 2次元バイナリマスクのリスト

    Returns:
      synapse マスク内に重心が含まれる vec マスクのインデックスリスト
    """
    indices = []
    centroids =[]
    for idx, vec_mask in enumerate(vec_masks):
        try:
            centroid = compute_centroid(vec_mask)
        except ValueError:
            # マスクに領域がない場合はスキップ
            continue

        # centroidは浮動小数点の (row, col) 座標なので整数に変換
        centroid_int = (int(round(centroid[0])), int(round(centroid[1])))

        # synapseマスク内の centroid の値を確認する
        # 画像外の座標の場合はスキップ
        if not (0 <= centroid_int[0] < synapse_mask.shape[0] and 0 <= centroid_int[1] < synapse_mask.shape[1]):
            continue

        if synapse_mask[centroid_int] > 0:
            indices.append(idx)
            centroids.append(centroid)
    return indices, centroids



def nearest_neighbor_distance(points):
    # 全ての点の組み合わせの距離を計算
    distances = cdist(points, points)
    
    # 自分自身との距離（0）を無視して最小値を取得
    # axis=1で各行の最小値を取得
    nn_distances = np.partition(distances, 1, axis=1)[:, 1]
    
    return nn_distances

def get_candidate_info(synapse_mask, az_masks, distance_threshold):
    """
    シナプスマスクから距離変換を計算し、
    各 az マスク内の最小距離がthreshold以下なら候補として採用する。
    
    Parameters:
      synapse_mask      : 2次元のシナプスバイナリマスク (0 or 1)
      az_masks          : 2次元のazバイナリマスクのリスト
      distance_threshold: 閾値（ピクセル単位）
      
    Returns:
      candidates: リスト
         各候補は辞書形式で { "index": i, "min_distance": d, "mask": az_mask } 
         を格納。
    """
    # distance_transform_edt はゼロとなる場所が特徴となるので、
    # シナプス領域を 1 から 0 に置き換えたものを用いる。
    # つまり、synapse_maskが1の部分が特長となるようにするには、1 - mask を渡す。
    dist_transform = distance_transform_edt(1 - synapse_mask)
    
    candidates = []
    for i, az in enumerate(az_masks):
        # az_mask 内のピクセルそれぞれについてシナプス領域までの距離を抽出
        if np.sum(az) == 0:
            # 有効な領域がなければスキップ
            continue
        min_dist = np.min(dist_transform[az.astype(bool)])
        print(i, min_dist)
        if min_dist <= distance_threshold:
            candidates.append({
                "index": i,
                "min_distance": min_dist,
                "mask": az.astype(bool)  # 重複判定用に bool 型に変更して保持する
            })
    return candidates

def get_candidate_info2(synapse_mask, az_masks, distance_threshold):
    """
    シナプスマスクから距離変換を計算し、
    各 az マスク内の最小距離がthreshold以下なら候補として採用する。
    
    Parameters:
      synapse_mask      : 2次元のシナプスバイナリマスク (0 or 1)
      az_masks          : 2次元のazバイナリマスクのリスト
      distance_threshold: 閾値（ピクセル単位）
      
    Returns:
      candidates: リスト
         各候補は辞書形式で { "index": i, "min_distance": d, "mask": az_mask } 
         を格納。
    """
    # distance_transform_edt はゼロとなる場所が特徴となるので、
    # シナプス領域を 1 から 0 に置き換えたものを用いる。
    # つまり、synapse_maskが1の部分が特長となるようにするには、1 - mask を渡す。
    dist_transform = distance_transform_edt(1 - synapse_mask)
    
    candidates = []
    for i, az in enumerate(az_masks):
        # az_mask 内のピクセルそれぞれについてシナプス領域までの距離を抽出
        if np.sum(az) == 0:
            # 有効な領域がなければスキップ
            continue
        min_dist = np.min(dist_transform[az.astype(bool)])
#         print(i, min_dist)
        if min_dist <= distance_threshold:
            candidates.append(i)
    return candidates

def union_find_init(n):
    """ Union-Find の初期化 """
    return list(range(n))

def union_find_find(parents, i):
    """ 親の検索 """
    if parents[i] != i:
        parents[i] = union_find_find(parents, parents[i])
    return parents[i]

def union_find_union(parents, i, j):
    """ 連結 """
    ri = union_find_find(parents, i)
    rj = union_find_find(parents, j)
    if ri != rj:
        parents[rj] = ri

def group_overlapping_candidates(candidates):
    """
    候補一覧（辞書形式）のうち、マスクの重複（少しでも共通領域があるか）によってグループ化し、
    各グループから代表候補（ここではシナプスまでの最小距離が小さいもの）を返す。
    
    Parameters:
      candidates: リスト{ "index":, "min_distance":, "mask": } の候補
    
    Returns:
      rep_indices: 代表候補の az マスクのインデックスリスト
    """
    n = len(candidates)
    if n == 0:
        return []
    
    parents = union_find_init(n)
    
    # 候補同士のマスクの重複が発生していないか確認
    for i in range(n):
        for j in range(i + 1, n):
            # もし、候補iと候補jで少しでも共通領域( True 同士)があればグループ化
            if np.any(np.logical_and(candidates[i]["mask"], candidates[j]["mask"])):
                union_find_union(parents, i, j)
    
    # グループごとに代表候補を決定
    groups = defaultdict(list)
    for idx in range(n):
        root = union_find_find(parents, idx)
        groups[root].append(candidates[idx])
    
    rep_indices = []
    for group in groups.values():
        # 代表候補として、シナプスまでの距離が最も小さい候補を選ぶ
        rep = min(group, key=lambda x: x["min_distance"])
        rep_indices.append(rep["index"])
    
    return rep_indices

def select_az_candidates(synapse_mask, az_masks, distance_threshold):
    """
    ・synapseマスクとシナプスまでの距離が閾値以下のazマスク画像を候補に抽出する。
    ・候補同士のマスク画像が少しでも重複している場合はグループ化し、代表1枚のみ採用する。
    
    Parameters:
      synapse_mask      : 2次元バイナリマスク (0,1)
      az_masks          : 2次元バイナリマスク画像のリスト
      distance_threshold: 距離の閾値
      
    Returns:
      rep_indices: 条件を満たす代表的な az マスクのインデックスリスト
    """
    # 1. シナプスとの距離が閾値以下の候補を抽出
    candidates = get_candidate_info(synapse_mask, az_masks, distance_threshold)
    # 2. 重複(オーバーラップ)している候補グループから代表を選定
    rep_indices = group_overlapping_candidates(candidates)
    return rep_indices

def calculate_major_axis_length(mask):
    """
    入力: mask - 2次元のバイナリマスク画像（0:背景, 1 or 255:物体）
    出力: 物体の長軸の長さ（pixels）
    
    ※ マスク画像内に複数のオブジェクトが存在する場合は、
      
       #面積が最大のオブジェクトの長軸長を返します。
    """
    # マスクが 255 を物体としている場合、0/1 の二値化
    binary_mask = (mask > 0).astype(np.uint8)

    # ラベル付け
    labeled_mask = label(binary_mask)
    regions = regionprops(labeled_mask)
    
    if len(regions) == 0:
        return 0  # 物体が見つからなかった場合
    
#     target_region = max(regions, key=lambda r: r.area)
    
    # 全リージョンの major_axis_length を合計
    total_major = sum(region.major_axis_length for region in regions)
    return total_major

def calc_az_vec_dists(az_mask, centroids):
    
    combined_az_mask = np.zeros((az_mask[0].shape[0], az_mask[0].shape[1]), dtype=bool)
    for mask in az_mask:
        combined_az_mask = np.logical_or(combined_az_mask, mask)
        
    distance_map = distance_transform_edt(~combined_az_mask)
    
    # --- 各 vec の重心座標に対して、az マスクまでの距離を計算 ---
    distances = []
    for centroid in centroids:
        # 重心座標は画像座標（row, col）の順になっていると仮定しています。
        # 小数点がある場合は、ここでは最近傍の整数画素 (round で丸める) を利用して距離をサンプリングします
        row = int(round(centroid[0]))
        col = int(round(centroid[1]))
        
        # 画像外にならないようにチェック（オプション）
        row = min(max(row, 0), distance_map.shape[0] - 1)
        col = min(max(col, 0), distance_map.shape[1] - 1)
        
        d = distance_map[row, col]
        distances.append(d)
    return distances


def labelme_json_to_instances(json_path, label0):
    # 1) LabelMe JSON 読み込み
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    H = data["imageHeight"]
    W = data["imageWidth"]

    boxes, classes, masks = [], [], []

    if label0 !="vec":
        label2id = {"az": 1, "synapse": 0}
    else:
        label2id = {"vesicle": 0}

    for shape in data["shapes"]:
        label = shape["label"]
        pts   = shape["points"]  # [[x, y], ...]

        # 2) バウンディングボックス計算
        # xs, ys = zip(*pts)
        # x0, y0, x1, y1 = min(xs), min(ys), max(xs), max(ys)
        # boxes.append([x0, y0, x1, y1])
        classes.append(label2id[label])

        # 3) マスク化：座標を整数に丸める
        int_pts = [(int(round(x)), int(round(y))) for x, y in pts]
        mask_img = Image.new("L", (W, H), 0)
        draw = ImageDraw.Draw(mask_img)
        draw.polygon(int_pts, outline=1, fill=1)
        m = np.array(mask_img, dtype=bool)
        masks.append(m)

    # 4) Detectron2 Instances に詰める
    inst = Instances((H, W))
    # inst.pred_boxes   = Boxes(torch.tensor(boxes, dtype=torch.float32))
    # inst.scores       = torch.ones(len(boxes), dtype=torch.float32)
    inst.pred_classes = torch.tensor(classes, dtype=torch.int64)
    mask_tensor = torch.stack(
        [torch.from_numpy(m.astype("uint8")) for m in masks], dim=0
    )
    inst.pred_masks = BitMasks(mask_tensor)

    return inst


def labelme_json_to_instances2(json_path):
    # 1) LabelMe JSON 読み込み
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    H = data["imageHeight"]
    W = data["imageWidth"]

    boxes, classes, masks = [], [], []

    
    label2id = {"az": 1, "synapse": 0, "vesicle":2}
  

    for shape in data["shapes"]:
        label = shape["label"]
        pts   = shape["points"]  # [[x, y], ...]

        # 2) バウンディングボックス計算
        # xs, ys = zip(*pts)
        # x0, y0, x1, y1 = min(xs), min(ys), max(xs), max(ys)
        # boxes.append([x0, y0, x1, y1])
        classes.append(label2id[label])

        # 3) マスク化：座標を整数に丸める
        int_pts = [(int(round(x)), int(round(y))) for x, y in pts]
        mask_img = Image.new("L", (W, H), 0)
        draw = ImageDraw.Draw(mask_img)
        draw.polygon(int_pts, outline=1, fill=1)
        m = np.array(mask_img, dtype=bool)
        masks.append(m)

    # 4) Detectron2 Instances に詰める
    inst = Instances((H, W))
    # inst.pred_boxes   = Boxes(torch.tensor(boxes, dtype=torch.float32))
    # inst.scores       = torch.ones(len(boxes), dtype=torch.float32)
    inst.pred_classes = torch.tensor(classes, dtype=torch.int64)
    mask_tensor = torch.stack(
        [torch.from_numpy(m.astype("uint8")) for m in masks], dim=0
    )
    inst.pred_masks = BitMasks(mask_tensor)

    return inst

# def run_experiment(val_name, params, run_id):
#     """
#     指定したパラメータで Detectron2 の学習・評価を実行する関数
#     Args:
#       params: 辞書型でハイパーパラメーター（例："SOLVER.BASE_LR", "SOLVER.MAX_ITER" など）
#       run_id: 実験識別用のID（出力フォルダなどに利用）
#     Returns:
#       bbox mAP と評価結果のディクショナリ
#     """
#     cfg = get_cfg()
#     # モデル設定のマージ
#     default_model = "COCO-InstanceSegmentation/mask_rcnn_X_101_32x8d_FPN_3x.yaml"
#     cfg.merge_from_file(model_zoo.get_config_file( params.get("MODEL", default_model)))
    
#     # 学習データセットの指定
#     cfg.DATASETS.TRAIN = (train_name_ori,)
#     cfg.DATASETS.TEST = ()  # 学習中はTestは設定しない
#     cfg.DATALOADER.NUM_WORKERS = 2

#     # モデル重みの初期化（Model Zoo の重みから）
#     cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url( params.get("MODEL", default_model))
    
#     # ハイパーパラメーターの適用
#     cfg.SOLVER.IMS_PER_BATCH = 2
#     cfg.SOLVER.BASE_LR =  float(params.get("SOLVER.BASE_LR", 0.00025))
#     cfg.SOLVER.MAX_ITER = int(params.get("SOLVER.MAX_ITER", 300))
#     cfg.SOLVER.WEIGHT_DECAY = float(params.get("SOLVER.WEIGHT_DECAY", 300))
#     cfg.SOLVER.STEPS = []  # 学習率の減衰は行わない
#     cfg.SOLVER.WARMUP_FACTOR = float(params.get("SOLVER.WARMUP_FACTOR", 300))
#     cfg.SOLVER.WARMUP_ITERS = int(params.get("SOLVER.WARMUP_ITERS", 300))
    
#     cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = int(params.get("MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE", 300))
#     # cfg.MODEL.ANCHOR_GENERATOR.SIZES = (params.get("MODEL.ANCHOR_GENERATOR.SIZES", 300)
#     cfg.MODEL.BACKBONE.FREEZE_AT = int(params.get("MODEL.BACKBONE.FREEZE_AT", 300))
#     cfg.MODEL.RPN.NMS_THRESH = float(params.get("MODEL.RPN.NMS_THRESH", 300))
#     cfg.MODEL.ROI_HEADS.NMS_THRESH_TEST = float(params.get("MODEL.ROI_HEADS.NMS_THRESH_TEST", 300))
    
    
#     cfg.MODEL.ROI_HEADS.NUM_CLASSES = class_num  # 分類するクラス数
    
#     # 出力先ディレクトリの設定（run_idごとに分ける）
#     custom_output_dir = os.path.join(wsl_pyf, "detectron2_trained_model_resize", f"{train_name}_run{run_id}")
#     os.makedirs(custom_output_dir, exist_ok=True)
#     cfg.OUTPUT_DIR = custom_output_dir

#     print(f"--- Run {run_id} start with params: {params} ---")
#     # 学習実行
#     trainer = DefaultTrainer(cfg)
#     trainer.resume_or_load(resume=False)
#     trainer.train()

#     # ----------------------------------------------------------------
#     # 学習済みモデルを用いて評価
#     # ----------------------------------------------------------------
#     # 評価用のデータセット登録（すでに登録済みの場合は名前を変更）
#     val_dataset_name = f"{val_name}_val"
#     try:
#         val_data_dir = os.path.join(wsl_pyf, "coco_dir_test_syn")
#         val_annotation_path = os.path.join(val_data_dir, "annotations.json")
#         register_coco_instances(val_dataset_name, {}, val_annotation_path, val_data_dir)
#     except:
#         traceback.print_exc()

#     # 評価用設定に変更
#     cfg.DATASETS.TEST = (val_dataset_name,)
#     cfg.MODEL.WEIGHTS = os.path.join(custom_output_dir, "model_final.pth")
#     cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    
#     # 予測器の作成
#     predictor = DefaultPredictor(cfg)
    
#     evaluator = COCOEvaluator(val_dataset_name, cfg, False, output_dir=custom_output_dir)
#     val_loader = build_detection_test_loader(cfg, val_dataset_name)
#     metrics = inference_on_dataset(predictor.model, val_loader, evaluator)
    
#     bbox_map = metrics.get("bbox", None)
#     print(f"--- Run {run_id} finished: bbox mAP = {bbox_map} ---")
#     return bbox_map, metrics


def create_mask_for_image(test_folder, json_file, target_file_name, target_category_id=0):
    """
    JSONファイル（COCO形式）の情報から、指定した画像ファイル名(target_file_name)
    に対応する画像の、target_category_id の全セグメンテーションを統合したマスクを生成します。
    
    Parameters:
      json_file (str): アノテーションJSONのファイルパス
      target_file_name (str): 対象画像のファイル名（例："JPEGImages/Keiko_9401_SD005.jpg"）
      target_category_id (int, optional): 対象のカテゴリID（デフォルトは 0）
      
    Returns:
      mask (PIL.Image): 統合マスク（モード "L", 0: 背景, 1: 塗りつぶし領域）
      img (PIL.Image): 対象画像（存在する場合）
      img_info (dict): 対象画像の情報（サイズ・IDなど）
    """
    # JSONファイルの読み込み
    with open(json_file, "r") as f:
#         print("open {}".format(json_file))
        data = json.load(f)
#     print(target_file_name)
    
   
    
    # 対象画像情報の検索（file_name の一致を条件）
    img_info = None
    for info in data["images"]:
#         print(target_file_name.endswith(info["file_name"]))
#         print(info["file_name"])
        # JSON内の file_name にパスが含まれている場合もあるので、末尾部分でチェック
#         if target_file_name.endswith(info["file_name"]) or info["file_name"] in target_file_name:
        if target_file_name in info["file_name"]:
            img_info = info
            break

    if img_info is None:
        raise ValueError(f"指定の画像ファイル名 {target_file_name} がJSON内に見つかりません。")
        
    # 画像の読み込み
    if os.path.exists(wsl_pyf+"/"+test_folder+"/JPEGImages/"+target_file_name):
        img = Image.open(wsl_pyf+"/"+test_folder+"/JPEGImages/"+target_file_name).convert("RGB")
    else:
        raise ValueError(f"画像ファイル {target_file_name} が存在しません。")
        
    width, height = img_info["width"], img_info["height"]
    
    # 空のマスク画像（モード "L"：グレースケール）を作成（初期値0＝背景）
    mask = Image.new("L", (width, height), 0)
    draw = ImageDraw.Draw(mask)
    
    # 対象画像のimage_idに一致し、かつ対象category_id のアノテーションを集めて統合する
    for ann in data["annotations"]:
        if ann["image_id"] != img_info["id"]:
            continue
        if ann["category_id"] != target_category_id:
            continue
        
        # segmentationはポリゴンの座標リストのリスト（例: [[x1, y1, x2, y2, ...]]）
        for seg in ann["segmentation"]:
            # ImageDraw.polygonは座標リスト形式 [x1, y1, x2, y2, …] を受け取るのでそのままでOK
            draw.polygon(seg, outline=1, fill=1)
            
    mask = np.array(mask)
    
    return mask

def compute_fscore(test_mask, target_mask):
    """
    二値マスクの ndarray (0,1) である test_mask と target_mask の F1スコアを計算する関数。
    
    Parameters:
      test_mask (ndarray): 評価対象のマスク
      target_mask (ndarray): 正解（ゴールドスタンダード）のマスク
      
    Returns:
      float: F1スコア
    """
    # フラット化して各画素の評価を行う
    test_flat = test_mask.flatten()
    target_flat = target_mask.flatten()
    
    # True Positive, False Positive, False Negativeの画素数を計算
    tp = np.sum((test_flat == 1) & (target_flat == 1))
    fn = np.sum((test_flat == 1) & (target_flat == 0))
    fp = np.sum((test_flat == 0) & (target_flat == 1))
    
    # 分母が0の場合のチェック（両方共マスクが全く1でなければ0とする）
    if (2 * tp + fp + fn) == 0:
        return 0.0
    
    # F1 score(=Fscore) の計算
    f1 = (2 * tp) / (2 * tp + fp + fn)
    return f1


def compute_fscore_all(test_mask, target_mask):
    """
    二値マスクの ndarray (0,1) である test_mask と target_mask の F1スコアを計算する関数。
    
    Parameters:
      test_mask (ndarray): 評価対象のマスク
      target_mask (ndarray): 正解（ゴールドスタンダード）のマスク
      
    Returns:
      float: F1スコア
    """
    # フラット化して各画素の評価を行う
    test_flat = test_mask.flatten()
    target_flat = target_mask.flatten()
    
    # True Positive, False Positive, False Negativeの画素数を計算
    tp = np.sum((test_flat == 1) & (target_flat == 1))
    tn = np.sum((test_flat == 0) & (target_flat == 0))
    fn = np.sum((test_flat == 1) & (target_flat == 0))
    fp = np.sum((test_flat == 0) & (target_flat == 1))
    
    # 分母が0の場合のチェック（両方共マスクが全く1でなければ0とする）
    if (2 * tp + fp + fn) == 0:
        return 0.0
    
    # F1 score(=Fscore) の計算
    f1 = (2 * tp) / (2 * tp + fp + fn)
    
    # Sensitivity (再現率, 真陽性率)
    # 定義: TP / (TP + FN)
    denom_sens = (tp + fn)
    sense = tp / denom_sens if denom_sens > 0 else 0.0

    # Specificity (特異度, 真陰性率)
    # 定義: TN / (TN + FP)
    denom_spec = (tn + fp)
    spe = tn / denom_spec if denom_spec > 0 else 0.0
    
    
   
    return f1, sense, spe



def run_experiment2(train_name_ori, val_name, params, run_id, overlap_del, batch_norm, ROI_Norm, area_th, mask_th):
    """
    指定したパラメータで Detectron2 の学習・評価を実行する関数
    Args:
      params: 辞書型でハイパーパラメーター（例："SOLVER.BASE_LR", "SOLVER.MAX_ITER" など）
      run_id: 実験識別用のID（出力フォルダなどに利用）
    Returns:
      bbox mAP と評価結果のディクショナリ
    """
    cfg = get_cfg()
    # モデル設定のマージ
    default_model = "COCO-InstanceSegmentation/mask_rcnn_X_101_32x8d_FPN_3x.yaml"
    
    if not "new_baselines" in params["MODEL"]:
        cfg.merge_from_file(model_zoo.get_config_file( params.get("MODEL", default_model)))
    else:
        cfg.merge_from_file(wsl_pyf + "/detectron2/configs/"+params["MODEL"])
    
    # 学習データセットの指定
    
    m = params["MODEL"]
#     lr = float(params["SOLVER.BASE_LR"])
#     max_iter = int(params["SOLVER.MAX_ITER"])
#     wd  =float(params["SOLVER.WEIGHT_DECAY"])
#     wf= float(params["SOLVER.WARMUP_FACTOR"])
#     wi = int(params["SOLVER.WARMUP_ITERS"])
#     bs = int(params["MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE"])
    # anchor = float(params["MODEL.ANCHOR_GENERATOR.SIZES":anchor,
#     freeze = int(params["MODEL.BACKBONE.FREEZE_AT"])
#     nms_th = float(params["MODEL.RPN.NMS_THRESH"])
#     ////
    
    
    

    
    
    cfg.DATASETS.TRAIN = (train_name_ori,)
    cfg.DATASETS.TEST = ()  # 学習中はTestは設定しない
    cfg.DATALOADER.NUM_WORKERS = 0

    # モデル重みの初期化（Model Zoo の重みから）
    cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url( params.get("MODEL", default_model))
    
    # ハイパーパラメーターの適用
    cfg.SOLVER.IMS_PER_BATCH =  params.get("IMS_PER_BATCH", 1)
    
    
    print("model", params["MODEL"])
#     print("wd", params["SOLVER.WEIGHT_DECAY"])
    if params["SOLVER.BASE_LR"] !=None:
        lr = float(params["SOLVER.BASE_LR"])
        cfg.SOLVER.BASE_LR =  float(params.get("SOLVER.BASE_LR", 0.00025))
#     else:
#         lr = None #params["SOLVER.BASE_LR"]
    
    if params["SOLVER.MAX_ITER"] != None:
        max_iter = int(params["SOLVER.MAX_ITER"])
        cfg.SOLVER.MAX_ITER = int(params.get("SOLVER.MAX_ITER", 300))
#     else:
#         max_iter = None
    
    if params["SOLVER.WEIGHT_DECAY"] != None:
        wd  =float(params["SOLVER.WEIGHT_DECAY"])
        cfg.SOLVER.WEIGHT_DECAY = float(params.get("SOLVER.WEIGHT_DECAY", 300))
#     else:
#         wd = None
        
    if params["SOLVER.WARMUP_FACTOR"] != None:
        wf = float(params["SOLVER.WARMUP_FACTOR"])
        cfg.SOLVER.WARMUP_FACTOR = float(params.get("SOLVER.WARMUP_FACTOR", 300))
#     cfg.SOLVER.STEPS = []  # 学習率の減衰は行わない


    if params["SOLVER.WARMUP_ITERS"] != None:
        wi = float(params["SOLVER.WARMUP_ITERS"])
        cfg.SOLVER.WARMUP_ITERS = int(params.get("SOLVER.WARMUP_ITERS", 300))
        
    
    if params["MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE"] != None:
        bs =int(params["MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE"])
        cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = int(params.get("MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE", 300))
        
    if params["MODEL.ANCHOR_GENERATOR.SIZES"] != None:
        anchor = params["MODEL.ANCHOR_GENERATOR.SIZES"]
        cfg.MODEL.ANCHOR_GENERATOR.SIZES = (params.get("MODEL.ANCHOR_GENERATOR.SIZES", 300))
                                            
    if params["MODEL.BACKBONE.FREEZE_AT"] != None:
        freeze = int(params["MODEL.BACKBONE.FREEZE_AT"])
        cfg.MODEL.BACKBONE.FREEZE_AT = int(params.get("MODEL.BACKBONE.FREEZE_AT", 300))
        
    if params["SOLVER.GAMMA"] != None:
        gamma = float(params["SOLVER.GAMMA"])
        cfg.SOLVER.GAMMA = float(params.get("SOLVER.GAMMA", 300))
        
    if params["SOLVER.STEPS"] != None:
        steps = params["SOLVER.STEPS"]
        cfg.SOLVER.STEPS = (params.get("SOLVER.STEPS", 300))
                                            
    if params["MODEL.RPN.NMS_THRESH"] != None:
        nms_th = float(params["MODEL.RPN.NMS_THRESH"])
        cfg.MODEL.RPN.NMS_THRESH = float(params.get("MODEL.RPN.NMS_THRESH", 300))
                                            
    if params["MODEL.ROI_HEADS.NMS_THRESH_TEST"] != None:
        nms_th_t = float(params["MODEL.ROI_HEADS.NMS_THRESH_TEST"])
        cfg.MODEL.ROI_HEADS.NMS_THRESH_TEST = float(params.get("MODEL.ROI_HEADS.NMS_THRESH_TEST", 300))
    
    if batch_norm != "FrozenBN":
        cfg.MODEL.BACKBONE.FROZEN_BN = "False"
    
    if not "new_baselines/mask_rcnn_regnety_4gf_dds_FPN_400ep_LSJ" in  params["MODEL"]:
        cfg.MODEL.RESNETS.NORM =batch_norm
        cfg.MODEL.FPN.NORM = batch_norm
        cfg.MODEL.ROI_HEADS.NORM = batch_norm
        
#     if batch_norm == "GN":
#         if "Misc" in params["MODEL"]:
#             nm_size = int(params["MODEL.GROUP_NORM.NUM_GROUPS"])
            
#             cfg.MODEL.GROUP_NORM.NUM_GROUPS = nm_size
            
#     if not "new_baselines" in params["MODEL"]:
#         cfg.MODEL.ROI_BOX_HEAD.NORM = ROI_Norm
            

#     if "new_baselines" in params["MODEL"]:
#         if params["MODEL.ROI_HEADS.POOLER_SAMPLING_RATIO"]!= None:
#             cfg.MODEL.ROI_HEADS.POOLER_SAMPLING_RATIO = int(params["MODEL.ROI_HEADS.POOLER_SAMPLING_RATIO"])
    

    cfg.MODEL.ROI_HEADS.NUM_CLASSES = class_num  # 分類するクラス数
    print("Gamma", cfg.SOLVER.GAMMA)
    print("steps", cfg.SOLVER.STEPS)
                                            
#train_name = train_name_ori + "/{}/lr{}_maxiter{}_wd{}_wf{}_wi{}_bs{}_freeze{}_nmsth{}_nmstht{}_RESnorm{}_ROInorm{}_GN{}_psr{}".format(m, cfg.SOLVER.BASE_LR, cfg.SOLVER.MAX_ITER, cfg.SOLVER.WEIGHT_DECAY, cfg.SOLVER.WARMUP_FACTOR, cfg.SOLVER.WARMUP_ITERS, cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE, cfg.MODEL.BACKBONE.FREEZE_AT, cfg.MODEL.RPN.NMS_THRESH, cfg.MODEL.ROI_HEADS.NMS_THRESH_TEST, batch_norm, ROI_Norm, cfg.MODEL.GROUP_NORM.NUM_GROUPS, cfg.MODEL.ROI_HEADS.POOLER_SAMPLING_RATIO)


#     train_name = train_name_ori + "/{}/lr{}_maxiter{}_wd{}_wf{}_wi{}_bs{}_freeze{}_nmsth{}_nmstht{}_RESnorm{}_ROInorm{}_GN{}_psr{}".format(m, cfg.SOLVER.BASE_LR, cfg.SOLVER.MAX_ITER, cfg.SOLVER.WEIGHT_DECAY, cfg.SOLVER.WARMUP_FACTOR, cfg.SOLVER.WARMUP_ITERS, cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE, cfg.MODEL.BACKBONE.FREEZE_AT, cfg.MODEL.RPN.NMS_THRESH, cfg.MODEL.ROI_HEADS.NMS_THRESH_TEST, batch_norm, ROI_Norm, cfg.MODEL.RESNETS.NORM, cfg.MODEL.ROI_HEADS.POOLER_SAMPLING_RATIO)
    
    if "Misc" in params["MODEL"]:
#         if params["IMS_PER_BATCH"] == 1:
#             train_name = train_name_ori + "/{}/lr{}_maxiter{}_wd{}_wf{}_wi{}_bs{}_freeze{}_nmsth{}_nmstht{}_RESnorm{}_ROInorm{}".format(m, cfg.SOLVER.BASE_LR, cfg.SOLVER.MAX_ITER, cfg.SOLVER.WEIGHT_DECAY, cfg.SOLVER.WARMUP_FACTOR, cfg.SOLVER.WARMUP_ITERS, cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE, cfg.MODEL.BACKBONE.FREEZE_AT, cfg.MODEL.RPN.NMS_THRESH, cfg.MODEL.ROI_HEADS.NMS_THRESH_TEST, cfg.MODEL.RESNETS.NORM, cfg.MODEL.ROI_BOX_HEAD.NORM)
#         else:
        train_name = train_name_ori + "/{}/lr{}_maxiter{}_wd{}_wf{}_wi{}_imsbs{}_bs{}_freeze{}_gamma{}_steps{}-{}_nmsth{}_nmstht{}_RESnorm{}_ROInorm{}".format(m, cfg.SOLVER.BASE_LR, cfg.SOLVER.MAX_ITER, cfg.SOLVER.WEIGHT_DECAY, cfg.SOLVER.WARMUP_FACTOR, cfg.SOLVER.WARMUP_ITERS, cfg.SOLVER.IMS_PER_BATCH,  cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE, cfg.MODEL.BACKBONE.FREEZE_AT, cfg.SOLVER.GAMMA, cfg.SOLVER.STEPS[0],cfg.SOLVER.STEPS[1], cfg.MODEL.RPN.NMS_THRESH, cfg.MODEL.ROI_HEADS.NMS_THRESH_TEST, cfg.MODEL.RESNETS.NORM, cfg.MODEL.ROI_BOX_HEAD.NORM)
      
    
    if "COCO" in params["MODEL"]:
        train_name = train_name_ori + "/{}/lr{}_maxiter{}_wd{}_wf{}_wi{}_bs{}_freeze{}_nmsth{}_nmstht{}_RESnorm{}_ROInorm{}".format(m, cfg.SOLVER.BASE_LR, cfg.SOLVER.MAX_ITER, cfg.SOLVER.WEIGHT_DECAY, cfg.SOLVER.WARMUP_FACTOR, cfg.SOLVER.WARMUP_ITERS, cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE, cfg.MODEL.BACKBONE.FREEZE_AT, cfg.MODEL.RPN.NMS_THRESH, cfg.MODEL.ROI_HEADS.NMS_THRESH_TEST, cfg.MODEL.RESNETS.NORM, cfg.MODEL.ROI_BOX_HEAD.NORM)
    if "new_baselines" in params["MODEL"]:
        if not "mask_rcnn_regnety_4gf_dds_FPN_400ep_LSJ":
            train_name = train_name_ori + "/{}/lr{}_maxiter{}_wd{}_wf{}_wi{}_bs{}_freeze{}_nmsth{}_nmstht{}_RESnorm{}_psr{}".format(m, cfg.SOLVER.BASE_LR, cfg.SOLVER.MAX_ITER, cfg.SOLVER.WEIGHT_DECAY, cfg.SOLVER.WARMUP_FACTOR, cfg.SOLVER.WARMUP_ITERS, cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE, cfg.MODEL.BACKBONE.FREEZE_AT, cfg.MODEL.RPN.NMS_THRESH, cfg.MODEL.ROI_HEADS.NMS_THRESH_TEST, cfg.MODEL.RESNETS.NORM, cfg.MODEL.ROI_HEADS.POOLER_SAMPLING_RATIO)
        else:
            train_name = train_name_ori + "/{}/lr{}_maxiter{}_wd{}_wf{}_wi{}_bs{}_freeze{}_nmsth{}_nmstht{}_psr{}".format(m, cfg.SOLVER.BASE_LR, cfg.SOLVER.MAX_ITER, cfg.SOLVER.WEIGHT_DECAY, cfg.SOLVER.WARMUP_FACTOR, cfg.SOLVER.WARMUP_ITERS, cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE, cfg.MODEL.BACKBONE.FREEZE_AT, cfg.MODEL.RPN.NMS_THRESH, cfg.MODEL.ROI_HEADS.NMS_THRESH_TEST, cfg.MODEL.ROI_HEADS.POOLER_SAMPLING_RATIO)
     
    
#     train_name = train_name_ori + "/{}/lr{}_maxiter{}_wd{}_wf{}_wi{}_bs{}_freeze{}_nmsth{}_nmstht{}".format(m, cfg.SOLVER.BASE_LR, cfg.SOLVER.MAX_ITER, cfg.SOLVER.WEIGHT_DECAY, cfg.SOLVER.WARMUP_FACTOR, cfg.SOLVER.WARMUP_ITERS, cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE, cfg.MODEL.BACKBONE.FREEZE_AT, cfg.MODEL.RPN.NMS_THRESH, cfg.MODEL.ROI_HEADS.NMS_THRESH_TEST)
#     print(train_name)
                                            
    
    # 出力先ディレクトリの設定（run_idごとに分ける）
    custom_output_dir = os.path.join(wsl_pyf, out_dir_name, f"{train_name}")
    if not os.path.exists(os.path.join(custom_output_dir, "model_final.pth")):                                
        os.makedirs(custom_output_dir, exist_ok=True)
        cfg.OUTPUT_DIR = custom_output_dir

        # === ここから再開ロジック追加 ===
        # 出力ディレクトリ内の model_*.pth（model_final.pth を除く）を列挙
        ckpts = sorted(glob.glob(os.path.join(cfg.OUTPUT_DIR, "model_*.pth")))
        # 最後に置かれたチェックポイントがあれば
        if ckpts:
            # model_final.pth は再開対象外
            ckpts = [p for p in ckpts if not p.endswith("model_final.pth")]
        if ckpts:
            last_ckpt = ckpts[-1]
            print(f"🔄 途中チェックポイントを発見 → {last_ckpt} から再開します")
            cfg.MODEL.WEIGHTS = last_ckpt
            resume_flag = True
        else:
            # 未学習 or 途中モデルなし
            print("▶️ 新規学習を開始します")
            cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
                params.get("MODEL", default_model)
            )
            resume_flag = False

        print(f"--- Run {train_name} start with params: {params} ---")
        # 学習実行
        trainer = DefaultTrainer(cfg)
    # resume=True なら last_ckpt を、False なら cfg.MODEL.WEIGHTS（初期重み）から
        trainer.resume_or_load(resume=resume_flag)
        trainer.train()
    else:                                        
        cfg.OUTPUT_DIR = custom_output_dir

    # ----------------------------------------------------------------
    # 学習済みモデルを用いて評価
    # ----------------------------------------------------------------
    # 評価用のデータセット登録（すでに登録済みの場合は名前を変更）
    val_dataset_name = f"{val_name}_val"
    try:
        val_data_dir = os.path.join(wsl_pyf, test_folder)
        val_annotation_path = os.path.join(val_data_dir, "annotations.json")
        register_coco_instances(val_dataset_name, {}, val_annotation_path, val_data_dir)
    except:
        traceback.print_exc()



    

    # 評価用設定に変更
    cfg.DATASETS.TEST = (val_dataset_name,)
    cfg.MODEL.WEIGHTS = os.path.join(custom_output_dir, "model_final.pth")
    cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    #     cfg.MODEL.DEVICE = "cpu"
    
    # 予測器の作成
    predictor = DefaultPredictor(cfg)

    #applied to test datasets
#     threshold =   #remove masks duplicated %
#     area_th = 300
    img_fol_name = test_folder#"coco_dir_test_syn_resize"
    img_fol = wsl_pyf + "/"+img_fol_name+"/JPEGImages/"
                                            
    if overlap_del == True:                           
        save_fol = custom_output_dir + "/"+ img_fol_name +"/"
        os.makedirs(save_fol, exist_ok=True)
    else:
        save_fol = custom_output_dir + "/"+ img_fol_name +"_overlap"+"/"
        os.makedirs(save_fol, exist_ok=True)                                     

    #img保存用
#     save_fol2 =os.path.join(win_pyf, "detectron2_trained_model_pre", f"{train_name}") + "/"+ img_fol_name +"/"
#     os.makedirs(save_fol2, exist_ok=True)
    f_score_li = []
    sense_li = []
    spe_li = []
    
    if not os.path.exists(custom_output_dir + "/"+ img_fol_name + "/fscore.csv") or not os.path.exists(custom_output_dir + "/"+ img_fol_name + "/sense.csv") or not os.path.exists(custom_output_dir + "/"+ img_fol_name + "/spe.csv"):
        
        for n, fname in enumerate(os.listdir(img_fol)):
            # if n>0:
            #     continue
            print(fname)
            im = cv2.imread(img_fol + "/"+fname) #tifffile.imread(imgPath) #
            # print(im.size)
            outputs = predictor(im)

            # outputs["instances"] を CPU に移動
            instances = outputs["instances"].to("cpu")

            # 各インスタンスのマスクとスコア，クラスを取得
            pred_masks = instances.pred_masks.cpu().numpy()  # shape: (N, H, W) ※ bool 型が望ましい
            scores = instances.scores.cpu().numpy()            # shape: (N,)
            pred_classes = instances.pred_classes.cpu().numpy()  # 各インスタンスのクラス（番号）


            if overlap_del ==True:
                # クラスごとにグループ分けして NMS を実施
                final_keep = []
                unique_classes = np.unique(pred_classes)
                for cls in unique_classes:
                    # 現在のクラスに属するインデックス取得
                    cls_indices = np.where(pred_classes == cls)[0]
                    kept = nms_mask_group(cls_indices, pred_masks, scores, mask_th)
                    final_keep.extend(kept)

                # フィルタリング後の Instances を作成
                instances = instances[final_keep]

            with open(save_fol + "/" + fname.replace(".jpg",".pkl"), 'wb') as f:
                pickle.dump(instances, f)

            # ----- 可視化（検出結果の重複が大きいものは除外済み）-----
            # bounding box の数に合わせた空文字列のリストを作成（ラベルの記述が不要な場合）
            empty_labels = [""] * len(instances)

            # Metadata の取得（学習時のデータセットのメタ情報）
            metadata = MetadataCatalog.get(cfg.DATASETS.TRAIN[0])


            v = Visualizer(im[:, :, ::-1], metadata, scale=0.8)

            # overlay_instances() を呼び出して、bounding box のみ描画（labels は空文字なので何も表示されない）
            v = v.overlay_instances(
                boxes=instances.pred_boxes,
                labels=empty_labels,
                assigned_colors=None  # または None のままでも OK。既定の色が使われる
            )
            # MetadataCatalog.get("mydataset").thing_classes = ["synapse", "az"]
            # v = Visualizer(im[:, :, ::-1], MetadataCatalog.get(cfg.DATASETS.TRAIN[0]), scale=0.8)

            # v = v.draw_instance_predictions(outputs["instances"].to("cpu"))
            # 画像を表示
            result_image = v.get_image()[:, :, ::-1]
            # cv2.imshow("result", result_image)

            # 画像をJPEG形式で保存

            cv2.imwrite(save_fol + "/" + fname, result_image)
    #         cv2.imwrite(save_fol2 + "/" + fname, result_image)

            # # 画像を表示するためのウィンドウを待機する
            # cv2.waitKey(0)
            # cv2.destroyAllWindows()


            #automatic validation
            test_mask = create_mask_for_image(test_folder, val_annotation_path, fname, 0)
            target_mask = np.max(np.array(instances.pred_masks), axis=0)
            f_score, sense, spe = compute_fscore_all(test_mask, target_mask)

    #         print("fscore: ", f_score)
            f_score_li.append(f_score)
            sense_li.append(sense)
            spe_li.append(spe)
            
        print(f_score_li)

        np.savetxt(custom_output_dir + "/"+ img_fol_name + "/fscore.csv", np.array(f_score_li), delimiter=",")
        np.savetxt(custom_output_dir + "/"+ img_fol_name + "/sense.csv", np.array(sense_li), delimiter=",")
        np.savetxt(custom_output_dir + "/"+ img_fol_name + "/spe.csv", np.array(spe_li), delimiter=",")
        
    
    # evaluator = COCOEvaluator(val_dataset_name, cfg, False, output_dir=custom_output_dir)
    # val_loader = build_detection_test_loader(cfg, val_dataset_name)
    # metrics = inference_on_dataset(predictor.model, val_loader, evaluator)
    
    # bbox_map = metrics.get("bbox", None)
    # print(f"--- Run {run_id} finished: bbox mAP = {bbox_map} ---")
    # return bbox_map, metrics
    
 # マスク描画用関数の例
def overlay_mask(mask, color, alpha_val=0.2):
    """
    mask: 2次元のバイナリ(0/1またはbool)の numpy 配列
    color: [R, G, B]（0-1の値）
    alpha_val: マスク領域の透明度
    """
    # RGBA画像用の配列を生成。初期値は 0（完全透明）
    colored_mask = np.zeros((*mask.shape, 4), dtype=np.float32)
    for c in range(3):
        # mask が True の位置に対して色を設定
        colored_mask[..., c] = mask * color[c]
    # アルファチャネルも mask 部分のみ alpha_val を設定。mask であれば alpha_val、そうでなければ 0
    colored_mask[..., 3] = mask * alpha_val
    return colored_mask
    
    
def detect_outliers_iqr(data, k=1.5, return_mask=False):
    """
    data: 1D array-like of numbers
    k: IQR multiplier (default 1.5). 大きくすると外れ値として検出されにくくなる。
    return_mask: True の場合、同長のブール配列（外れ値=True）も返す。

    Returns:
      outliers: numpy array of detected outlier values
      (optional) mask: boolean numpy array, True for outliers
    """
    arr = np.asarray(data, dtype=float)
    # 欠損値を除外して四分位を計算
    valid = ~np.isnan(arr)
    vals = arr[valid]
    if vals.size == 0:
        if return_mask:
            return np.array([]), np.zeros_like(arr, dtype=bool)
        return np.array([])

    q1 = np.percentile(vals, 25)
    q3 = np.percentile(vals, 75)
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr

    mask = (arr < lower) | (arr > upper)
    # 欠損値は外れ値と見なさない
    mask = mask & valid

    outliers = arr[mask]
    if return_mask:
        return outliers, mask
    return outliers

In [38]:
#analyze vesicles and presynapses   by samples  from one folder
distance_threshold = 5
vecnum_in_syn = 10
scale = 1.7  #nm/pixel
# syn_fol =  "coco_dir_ex_syn"
# vec_fol =  "coco_dir_ex_vec"
# model_syn_name = "syn_az_aug"
# model_vec_name = "vec_aug"

save_fol = wsl_pyf + "/analyzed_snap2/"
os.makedirs(save_fol , exist_ok=True)
unit = "vecall"

syn_fol = wsl_pyf+"/"+ "apply_json_snap2"#.format("syn")
# vec_fol = wsl_pyf+"/"+ "labeled_all_{}".format(unit)
ins_fol = "apply_json_snap_ins2"

os.makedirs(wsl_pyf + "/"+ ins_fol +"/", exist_ok=True)
# os.makedirs(wsl_pyf + "/"+"labeled_all_{}_ins".format(unit) +"/", exist_ok=True)


fname_lists = ["A_0","B_0","C_0","D_0","E_0","F_0","G_0","H_0"] 
fname_lists2 = ["A","B","C","D","E","F","G","H"] 
# syn_vec_num_total_ctrl = []
# syn_vec_area_total_ctrl = []
# syn_vec_NN_total_ctrl = []

# syn_vec_num_sarea_total_ctrl = []
# syn_vec_area_sarea_total_ctrl = []
# syn_vec_NN_sarea_total_ctrl = []

# syn_az_len_total_ctrl = []
# az_vec_dists_total_ctrl = []
vec_sum_total = []
syn_vec_num_total = []
syn_vec_area_total = []
syn_vec_NN_total = []
syn_az_len_total = []
syn_az_vec_dists_total = []

m_vec_NN_total=[]
m_az_vec_dists_li_total=[]

m_syn_vec_num_total = []
m_syn_vec_area_total = []
m_syn_vec_NN_total = []
m_syn_az_len_total = []
m_syn_az_vec_dists_total = []

# syn_vec_num_sarea_total = []
# syn_vec_area_sarea_total = []
# syn_vec_NN_sarea_total = []



for m, fname_pre in enumerate(fname_lists):
#     if m>0:
#         continue
    print(fname_pre)
    save_fol_file = save_fol + "/" +fname_pre +"/"
    os.makedirs(save_fol_file , exist_ok=True)
    
    if os.path.exists(save_fol_file  +"az_vec_dists_li.npy"):
        continue
    
    
    syn_vec_num = []   #vesicle number per synapse
    syn_vec_area = []   #vesicle area per synapse
    syn_vec_NN = []     #vesicle NN per synapse
    vec_NN = []
    
    syn_az_len = []
    syn_az_vec_dists = []
    az_vec_dists_li = []

    m_syn_vec_area = [] 
    m_syn_vec_NN = []
    m_syn_az_vec_dists = []
#     syn_vec_num_sarea = []
#     syn_vec_area_sarea = []
#     syn_vec_NN_sarea = []
    

    for n, fname in enumerate(os.listdir(syn_fol)):
        try:

            if not fname_pre in fname:
                continue
        #     if n >0:
        #         continue
            
            
            
            
            
            if os.path.exists(wsl_pyf + "/"+ins_fol +"/"+ fname.replace("json", "pkl")):
                with open(wsl_pyf + "/"+ins_fol +"/"+ fname.replace("json", "pkl"), "rb") as f:
                    instances = pickle.load(f)
            else:

                instances = labelme_json_to_instances2(syn_fol + "/" + fname)
                with open(wsl_pyf + "/"+ins_fol +"/"+ fname.replace("json", "pkl"), "wb") as f:
                    pickle.dump(instances, f)


    #         print(syn_instances)
    #         print(vec_instances)


            syn_ind= np.where(np.array(instances.pred_classes) == 0)[0]
            az_ind= np.where(np.array(instances.pred_classes) == 1)[0] 
            vec_ind= np.where(np.array(instances.pred_classes) == 2)[0] 
            print("syn_ind", len(syn_ind))
            print("az_ind", len(az_ind))
            print("vec_ind", len(vec_ind))
            syn_masks = np.array(instances.pred_masks.tensor.cpu().numpy()[syn_ind])
            az_masks = np.array(instances.pred_masks.tensor.cpu().numpy()[az_ind])
            
            
            #indivisual file param
            syn_az_len_i=[]
            syn_vec_num_i=[]
            syn_vec_area_i=[]
            syn_vec_NN_i=[]
            syn_az_vec_dists_i=[]

            m_syn_az_vec_dists_i=[]
            m_syn_vec_area_i=[]
            m_syn_vec_NN_i=[]
            
            vec_NN_i=[]
            az_vec_dists_li_i =[]


            for m, syn_mask in enumerate(syn_masks):
                # if m!=1:
                #     continue
                try:
                    #find az in a synapse
                    candidates = get_candidate_info(syn_mask, az_masks, distance_threshold)
                    syn_az_ind = group_overlapping_candidates(candidates)
                    az_mask = np.array(az_masks[syn_az_ind])
                    az_length = np.array(calculate_major_axis_length(az_mask))*scale

                    #analyze vesicles in a synapse

        #             vec_ind= np.where(np.array(vec_instances.pred_classes) == 0)[0] 
                    # print("vec_ind", vec_ind)

                    vec_masks = instances.pred_masks.tensor.cpu().numpy()[vec_ind]

                    vec_syn_ind, centroids = extract_vec_indices_by_synapse(syn_mask, vec_masks)
                    print("vec_syn_ind", vec_syn_ind)

                    if len(vec_syn_ind)> vecnum_in_syn:
            #             plt.imshow(np.max(np.array(vec_masks[vec_syn_ind]), axis=0))
            #             plt.show()
            #             plt.imshow(np.max(np.array(az_mask), axis=0))
            #             plt.show()

            #             syn_area = np.sum(np.array(syn_mask))
                        dists = np.array(nearest_neighbor_distance(centroids))*scale
                        az_vec_dists = np.array(calc_az_vec_dists(az_mask, centroids))*scale

                        vec_masks = vec_masks[vec_syn_ind]
                        vec_num = len(vec_masks)
                        vec_areas = np.array([np.sum(np.array(mask)) for mask in vec_masks])*scale*scale

#                         plt.hist(vec_areas)


                        syn_az_len.append(az_length)

                        syn_vec_num.append(vec_num)
                        syn_vec_area.append(vec_areas)
                        syn_vec_NN.append(dists)
                        syn_az_vec_dists.append(az_vec_dists)

                        m_syn_az_vec_dists.append(np.median(az_vec_dists))
                        m_syn_vec_area.append(np.median(vec_areas))
                        m_syn_vec_NN.append(np.median(dists ))
                        vec_NN += list(dists)
                        az_vec_dists_li += list(az_vec_dists)
                        
                        
                        syn_az_len_i.append(az_length)
                        syn_vec_num_i.append(vec_num)
                        syn_vec_area_i.append(vec_areas)
                        syn_vec_NN_i.append(dists)
                        syn_az_vec_dists_i.append(az_vec_dists)

                        m_syn_az_vec_dists_i.append(np.median(az_vec_dists))
                        m_syn_vec_area_i.append(np.median(vec_areas))
                        m_syn_vec_NN_i.append(np.median(dists ))
                        
                        vec_NN_i += list(dists)
                        az_vec_dists_li_i += list(az_vec_dists)

                        
                    
                except:
                    traceback.print_exc()
            #save indivisual param
            np.save(save_fol_file + fname.replace(".json", "") +"_syn_az_len", np.array(syn_az_len_i, dtype=object))
            np.save(save_fol_file + fname.replace(".json", "") +"_syn_vec_num", np.array(syn_vec_num_i, dtype=object))
            np.save(save_fol_file + fname.replace(".json", "") +"_syn_vec_area", np.array(syn_vec_area_i, dtype=object))
            np.save(save_fol_file + fname.replace(".json", "") +"_syn_vec_NN", np.array(syn_vec_NN_i, dtype=object))
            np.save(save_fol_file + fname.replace(".json", "") +"_syn_az_vec_dists", np.array(syn_az_vec_dists_i, dtype=object))
            np.save(save_fol_file + fname.replace(".json", "") +"_m_syn_az_vec_dists", np.array(m_syn_az_vec_dists_i, dtype=object))
            np.save(save_fol_file + fname.replace(".json", "") +"_m_syn_vec_area", np.array(m_syn_vec_area_i, dtype=object))
            np.save(save_fol_file + fname.replace(".json", "") +"_m_syn_vec_NN", np.array(m_syn_vec_NN_i, dtype=object))
            np.save(save_fol_file + fname.replace(".json", "") +"_vec_NN", np.array(vec_NN_i, dtype=object))
            np.save(save_fol_file + fname.replace(".json", "") +"_az_vec_dists_li", np.array(az_vec_dists_li_i, dtype=object))
            
        #             syn_vec_num_sarea.append(vec_num/syn_area)
        #             syn_vec_area_sarea.append(np.mean(vec_areas)/syn_area)
        #             syn_vec_NN_sarea.append(np.mean(dists )/syn_area)
        except:
                traceback.print_exc()
    
    #save sample total data
    np.save(save_fol_file  +"syn_az_len", np.array(syn_az_len, dtype=object))
    np.save(save_fol_file  +"syn_vec_num", np.array(syn_vec_num, dtype=object))
    np.save(save_fol_file  +"syn_vec_area", np.array(syn_vec_area, dtype=object))
    np.save(save_fol_file  +"syn_vec_NN", np.array(syn_vec_NN, dtype=object))
    np.save(save_fol_file  +"syn_az_vec_dists", np.array(syn_az_vec_dists, dtype=object))
    np.save(save_fol_file  +"m_syn_az_vec_dists", np.array(m_syn_az_vec_dists, dtype=object))
    np.save(save_fol_file  +"m_syn_vec_area", np.array(m_syn_vec_area, dtype=object))
    np.save(save_fol_file  +"m_syn_vec_NN", np.array(m_syn_vec_NN, dtype=object))
    np.save(save_fol_file  +"vec_NN", np.array(vec_NN, dtype=object))
    np.save(save_fol_file  +"az_vec_dists_li", np.array(az_vec_dists_li, dtype=object))
    
    print(save_fol_file  +"m_syn_az_vec_dists", np.array(m_syn_az_vec_dists))
        
    vec_sum = np.sum(syn_vec_num)
    
    m_syn_vec_num = np.mean(syn_vec_num)
    m_m_syn_vec_area = np.mean(m_syn_vec_area)  #list  #vesicle area per synapse
    m_m_syn_vec_NN = np.mean(m_syn_vec_NN)    #list #vesicle NN per synapse
    
    
    m_syn_az_len = np.mean(syn_az_len)
    m_m_syn_az_vec_dists = np.mean(m_syn_az_vec_dists) #list


    print(fname_pre)
    print("mean vesicle num", m_syn_vec_num)
    print("vesicle sum", vec_sum)
    print("mean vesicle area", m_m_syn_vec_area)
    print("mean vesicle NN per syn", m_m_syn_vec_NN)
    print("mean vesicle NN", np.mean(vec_NN))
    print("mean az length", m_syn_az_len)
    print("mean distance between az and vesicle per syn", m_m_syn_az_vec_dists)
    print("mean distance between az and vesicle", np.mean(az_vec_dists_li))
    
#     syn_vec_num_total += syn_vec_num
#     syn_vec_area_total+=syn_vec_area
#     syn_vec_NN_total+=syn_vec_NN
#     syn_az_len_total += syn_az_len
    vec_sum_total.append(vec_sum)
    m_syn_vec_num_total.append(m_syn_vec_num) 
    m_syn_vec_area_total.append(m_m_syn_vec_area) 
    m_syn_vec_NN_total.append(m_m_syn_vec_NN) 
    m_syn_az_len_total.append(m_syn_az_len) 
    m_syn_az_vec_dists_total.append(m_m_syn_az_vec_dists) 
    
    m_vec_NN_total.append(np.mean(vec_NN))
    m_az_vec_dists_li_total.append(np.mean(az_vec_dists_li))

    # syn_vec_num_sarea_total+=syn_vec_num_sarea
    # syn_vec_area_sarea_total+=syn_vec_area_sarea
    # syn_vec_NN_sarea_total+=syn_vec_NN_sarea

    # syn_az_len_total += syn_az_len

# print(syn_az_len_total)
# plt.hist(az_vec_dists_total)
# plt.show()

# plt.figure(figsize=(6, 8))
# plt.bar(fname_lists, m_syn_vec_num_total)
# plt.show()

A_0


/tmp/ipykernel_40/4048031810.py:110: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  syn_ind= np.where(np.array(instances.pred_classes) == 0)[0]
/tmp/ipykernel_40/4048031810.py:111: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  az_ind= np.where(np.array(instances.pred_classes) == 1)[0]
/tmp/ipykernel_40/4048031810.py:112: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keywor

syn_ind 5
az_ind 5
vec_ind 224
0 0.0
1 1924.644642524952
2 903.230313928845
3 1468.6592525157087
4 28.319604517012593
vec_syn_ind [0, 1, 2, 3, 4, 6, 7, 10, 17, 18, 19, 21, 24, 25, 26, 27, 28, 29, 31, 32, 33, 34, 36, 37, 39, 41, 42, 43, 47, 50, 52, 53, 57, 58, 60, 61, 62, 63, 66, 68, 69, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 221, 222, 223]
0 1862.7619278909476
1 0.0
2 899.4759585447517
3 1250.9996003196804
4 2103.9638780169207
vec_syn_ind [8, 9, 11, 12, 13, 15, 20, 22, 35, 38, 44, 45, 46, 49, 51, 54, 55, 67, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 164, 165, 166, 167, 168]
0 842.406077850819
1 816.169712743618
2 0.0
3 707.0077793065647
4 996.8289722916364
vec_syn_ind [40, 48, 56, 59, 64, 65, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102,

/tmp/ipykernel_40/4048031810.py:110: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  syn_ind= np.where(np.array(instances.pred_classes) == 0)[0]
/tmp/ipykernel_40/4048031810.py:111: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  az_ind= np.where(np.array(instances.pred_classes) == 1)[0]
/tmp/ipykernel_40/4048031810.py:112: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keywor

syn_ind 5
az_ind 6
vec_ind 251
0 0.0
1 1259.7956183444996
2 2442.005118749754
3 91.0494371207203
4 2398.9835347496655
5 2279.232546275171
vec_syn_ind [6, 16, 22, 30, 36, 40, 45, 47, 49, 54, 71, 78, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 242, 243, 244, 245]
0 1159.8318843694547
1 0.0
2 1054.9635064778308
3 1457.339013407656
4 949.2144120271247
5 824.1899053009568
vec_syn_ind [3, 11, 18, 39, 44, 46, 56, 62, 66, 67, 68, 150, 151, 152, 153, 154, 155, 156, 157, 158, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241]
0 2465.4297799775195
1 1154.1871598661978
2 1.4142135623730951
3 2751.491413760908
4 324.6166970443757
5 373.2264728017025
vec_syn_ind [0, 1, 2, 4, 5, 8, 9, 19, 20, 32, 41, 58, 59, 72, 74, 76, 79, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209]
0 232.57256931977167
1 1559.6

/tmp/ipykernel_40/4048031810.py:110: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  syn_ind= np.where(np.array(instances.pred_classes) == 0)[0]
/tmp/ipykernel_40/4048031810.py:111: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  az_ind= np.where(np.array(instances.pred_classes) == 1)[0]
/tmp/ipykernel_40/4048031810.py:112: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keywor

... (309 outputs left out)

/tmp/ipykernel_40/4048031810.py:110: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  syn_ind= np.where(np.array(instances.pred_classes) == 0)[0]
/tmp/ipykernel_40/4048031810.py:111: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  az_ind= np.where(np.array(instances.pred_classes) == 1)[0]
/tmp/ipykernel_40/4048031810.py:112: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keywor

syn_ind 6
az_ind 8
vec_ind 598
0 0.0
1 1765.7647068621573
2 1305.1302616980422
3 611.432743643976
4 575.9253423838892
5 1270.1181047445941
6 1301.7104132640254
7 781.1081871290302
vec_syn_ind [8, 34, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 412, 413, 414, 415, 416]
0 1404.5743127367807
1 699.4769474400139
2 2340.4617065869716
3 888.005067553108
4 773.7661145333259
5 0.0
6 0.0
7 537.3453265824502
vec_syn_ind [2, 11, 18, 21, 22, 23, 30, 39, 45, 53, 64, 73, 74, 76, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 17

/tmp/ipykernel_40/4048031810.py:110: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  syn_ind= np.where(np.array(instances.pred_classes) == 0)[0]
/tmp/ipykernel_40/4048031810.py:111: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  az_ind= np.where(np.array(instances.pred_classes) == 1)[0]
/tmp/ipykernel_40/4048031810.py:112: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keywor

syn_ind 7
az_ind 9
vec_ind 458
0 1.0
1 1633.1885990295182
2 1947.9294135055304
3 1780.6024261468365
4 856.5068592836838
5 1983.938003063604
6 1305.0003831417062
7 593.3776200700529
8 441.54274991216874
vec_syn_ind [0, 4, 5, 8, 11, 12, 13, 19, 22, 26, 27, 28, 36, 40, 46, 47, 48, 52, 53, 57, 58, 60, 63, 64, 67, 75, 76, 78, 79, 83, 84, 92, 94, 305, 306, 307, 308, 309, 310, 311, 312, 313, 314, 315, 316, 317, 318, 319, 320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354, 355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 367, 368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 378, 379, 380, 381, 382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 394, 395, 396, 397, 398, 399, 400, 401, 402, 403]
0 1625.1830666112664
1 0.0
2 373.4059453195677
3 427.873813173931
4 1107.3039329831715
5 143.69411957348845
6 242.33035303073365
7 958.2817957156444
8 1149.198851

In [ ]:
#save data
save_fol_all = wsl_pyf + "/analyzed_snap2/all/"
os.makedirs(save_fol_all , exist_ok=True)



In [21]:

vec_area_total =list(itertools.chain.from_iterable(syn_vec_area))
out, mask = detect_outliers_iqr(vec_area_total, return_mask=True)
print("outliers:", out)

outliers: [3690.53 2265.76 2739.72 2207.96 2262.87 2439.16 3161.66 2294.66 4655.79
 2222.41 2346.68 2262.87 5921.61 2863.99 3855.26 2242.64 2410.26 3112.53
 2577.88 3161.66 2378.47 7571.8  4652.9  3765.67 2557.65 3034.5  2768.62
 2589.44 2372.69 2950.69 4459.27 2537.42 2239.75 2355.35 2288.88 2228.19
 2589.44 2372.69 2314.89 2583.66 7002.47 2751.28 2222.41 4248.3  2401.59
 2343.79 2407.37 2673.25 5025.71 2332.23 2650.13 2251.31 2994.04 2213.74
 2462.28 2314.89 2378.47 2595.22 2303.33 2242.64 2528.75 2488.29 2473.84
 2800.41 2225.3  2690.59 2479.62 2638.57 2410.26 2511.41 2548.98 3008.49
 4395.69 2962.25 3861.04 4491.06 2453.61 2719.49 2277.32 4597.99 2459.39
 2459.39 2193.51 2687.7  2257.09 5860.92 4002.65 2193.51 2361.13 3326.39
 2430.49 2401.59 2583.66 2352.46 2494.07 2355.35 2384.25 2470.95 2531.64
 2291.77 6282.86 2274.43 2262.87 2262.87 2398.7  4080.68 2358.24 2476.73
 2314.89 2358.24 2207.96 4069.12 2942.02 2910.23 2725.27 3580.71 2387.14
 4051.78 2895.78 2849.54 4152.93  219.64 

In [5]:
#analyze vesicles and presynapses  get params for snap K40E samples
distance_threshold = 5
vecnum_in_syn = 10
# syn_fol =  "coco_dir_ex_syn"
# vec_fol =  "coco_dir_ex_vec"
# model_syn_name = "syn_az_aug"
# model_vec_name = "vec_aug"
unit = "vecall0"

save_fol = wsl_pyf + "/analyzed_snap2/"
os.makedirs(save_fol , exist_ok=True)
unit = "vecall"

syn_fol = wsl_pyf+"/"+ "apply_json_snap2"#.format("syn")
# vec_fol = wsl_pyf+"/"+ "labeled_all_{}".format(unit)
ins_fol = "apply_json_snap_ins2"
# os.makedirs(wsl_pyf + "/"+"labeled_all_{}_ins".format(unit) +"/", exist_ok=True)


fname_lists = ["E_0","F_0","G_0","H_0"] 
# fname_lists = ["#10"] 
# syn_vec_num_total_ctrl = []
# syn_vec_area_total_ctrl = []
# syn_vec_NN_total_ctrl = []

# syn_vec_num_sarea_total_ctrl = []
# syn_vec_area_sarea_total_ctrl = []
# syn_vec_NN_sarea_total_ctrl = []

# syn_az_len_total_ctrl = []
# az_vec_dists_total_ctrl = []

syn_vec_num_total = []
# sample_syn_vec_num_total = []

az_len_total = []
# sample_az_len_total = []

vec_area_total = []
# sample_vec_area_total = []
syn_vec_area_total = []
# sample_syn_vec_area_total = []

vec_NN_total = []
syn_vec_NN_total = []
az_vec_dists_total = []
syn_az_vec_dists_total = []
syn_az_vec_dists_ind_li = []
syn_NN_ind_li=[]
#by sample
# sample_vec_num_total = []
# sample_m_syn_az_vec_dists_total = []
# sample_syn_vec_num_cv_total= []
# sample_m_syn_az_vec_dists_cv_total= []
sample_syn_vec_num_total = []
sample_az_len_total = []
sample_syn_vec_area_total = []
sample_vec_area_total = []
# sample_syn_vec_area_total = []

sample_vec_NN_total = []
sample_syn_vec_NN_total = []
sample_az_vec_dists_total = []
sample_syn_az_vec_dists_total = []


# m_syn_vec_num_total = []
# m_syn_vec_area_total = []
# m_syn_vec_NN_total = []
# m_syn_az_len_total = []
# m_syn_az_vec_dists_total = []

# syn_vec_num_sarea_total = []
# syn_vec_area_sarea_total = []
# syn_vec_NN_sarea_total = []



for m, fname_pre in enumerate(fname_lists):
#     if m>0:
#         continue
    print(fname_pre)
    
    save_fol_file = save_fol + "/" +fname_pre +"/"
    
    if os.path.exists(save_fol_file  +"syn_az_len.npy"):
    
        syn_az_len =list(np.load(save_fol_file  +"syn_az_len.npy" , allow_pickle=True))
        syn_vec_num = list(np.load(save_fol_file  +"syn_vec_num.npy", allow_pickle=True))
        syn_vec_area =list(np.load(save_fol_file  +"syn_vec_area.npy", allow_pickle=True))
        syn_vec_NN =list(np.load(save_fol_file  +"syn_vec_NN.npy", allow_pickle=True))
        syn_az_vec_dists =list(np.load(save_fol_file  +"syn_az_vec_dists.npy", allow_pickle=True))
        m_syn_az_vec_dists =list(np.load(save_fol_file  +"m_syn_az_vec_dists.npy", allow_pickle=True))
        m_syn_vec_area = list(np.load(save_fol_file  +"m_syn_vec_area.npy", allow_pickle=True))
        m_syn_vec_NN =list(np.load(save_fol_file  +"m_syn_vec_NN.npy", allow_pickle=True))
        vec_NN =list(np.load(save_fol_file  +"vec_NN.npy", allow_pickle=True))
        az_vec_dists_li =list(np.load(save_fol_file  +"az_vec_dists_li.npy", allow_pickle=True))
        
        syn_num = len(syn_az_len)
        
        
        syn_vec_num_total += syn_vec_num
        print(len(syn_vec_num))
        az_len_total += syn_az_len 
        vec_area_total += syn_vec_area
        syn_vec_area_total += m_syn_vec_area
        vec_NN_total += vec_NN
        syn_vec_NN_total += m_syn_vec_NN
        az_vec_dists_total += az_vec_dists_li 
        print(len(az_vec_dists_li ))
        syn_az_vec_dists_total +=  m_syn_az_vec_dists
        
        syn_az_vec_dists_ind_li += syn_az_vec_dists
        syn_NN_ind_li += syn_vec_NN
        
        
        #by sample
        sample_syn_vec_num_total.append(syn_vec_num)
        sample_az_len_total.append(syn_az_len)
        sample_vec_area_total.append(syn_vec_area)
        sample_syn_vec_area_total.append(m_syn_vec_area)
        sample_vec_NN_total.append(vec_NN)
        sample_syn_vec_NN_total.append(m_syn_vec_NN)
        sample_az_vec_dists_total.append(az_vec_dists_li)
        sample_syn_az_vec_dists_total.append(m_syn_az_vec_dists)
        
                
#         sample_vec_sum_total.append(int(np.sum(syn_vec_num)/syn_num))
#         sample_syn_vec_num_cv_total.append(np.std(syn_vec_num)/np.mean(syn_vec_num))
#         sample_m_syn_az_vec_dists_total.append(np.mean(m_syn_az_vec_dists))
#         sample_m_syn_az_vec_dists_cv_total.append(np.std(m_syn_az_vec_dists)/np.mean(m_syn_az_vec_dists))
        
      
   

E_0
128
5741
F_0
124
6044
G_0
123
9570
H_0
132
9218


In [8]:
len(syn_vec_NN)

132

In [9]:
save_fol_file  +"az_vec_dists_li.npy"

'/home/gpu_data/data8/kinoshita_cfos/EM_analysis//analyzed0_med//#50/az_vec_dists_li.npy'

In [7]:
def to_jsonable(x):
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return float(x)
    if isinstance(x, list):
        return [to_jsonable(v) for v in x]
    if isinstance(x, tuple):
        return [to_jsonable(v) for v in x]  # JSONはtupleがないのでlistへ
    if isinstance(x, dict):
        return {str(k): to_jsonable(v) for k, v in x.items()}
    return x

In [8]:
import json
from itertools import chain

savefol = wsl_pyf + "/WT-vs-K40E/"
os.makedirs(savefol, exist_ok = True)

np.savetxt(savefol + "vec_num_per_synapse_K40E.csv", np.array(syn_vec_num_total), delimiter=",")
np.savetxt(savefol + "az_length_per_synapse_K40E.csv", np.array(az_len_total), delimiter=",")
np.savetxt(savefol + "vec_area_per_synapse_K40E.csv", np.array(syn_vec_area_total), delimiter=",")
np.savetxt(savefol + "vec_area_all_K40E.csv", np.array(list(chain.from_iterable(vec_area_total))), delimiter=",")

with open(savefol +"vec_area_per_synapse_all_K40E.json", "w", encoding="utf-8") as f:
    json.dump(to_jsonable(vec_area_total), f, ensure_ascii=False)

with open(savefol +"vec_NN_per_synapse_all_K40E.json", "w", encoding="utf-8") as f:
    json.dump(to_jsonable(syn_NN_ind_li), f, ensure_ascii=False)
    
with open(savefol +"az_vec_distance_per_synapse_all_K40E.json", "w", encoding="utf-8") as f:
    json.dump(to_jsonable(syn_az_vec_dists_ind_li), f, ensure_ascii=False)
    
np.savetxt(savefol + "vec_NN_per_synapse_K40E.csv", np.array(syn_vec_NN_total), delimiter=",")
np.savetxt(savefol + "vec_NN_all_K40E.csv", np.array(vec_NN_total), delimiter=",")
np.savetxt(savefol + "az_vec_distance_per_synapse_K40E.csv", np.array(syn_az_vec_dists_total), delimiter=",")
np.savetxt(savefol + "az_vec_distance_all_K40E.csv", np.array(az_vec_dists_total), delimiter=",")

In [9]:
#by samples   SD
import json
from itertools import chain

fname_lists = ["E_0","F_0","G_0","H_0"] 

savefol = wsl_pyf + "/WT-vs-K40E_sample/"
os.makedirs(savefol, exist_ok = True)

for i, fname in enumerate(fname_lists):
    
    if fname != "#12":
            fname = fname[0]
            
    print(fname)
    savefol_f = wsl_pyf + "/WT-vs-K40E_sample/" + fname +"/"
    os.makedirs(savefol_f, exist_ok = True)
    

    np.savetxt(savefol_f + "vec_num_per_synapse_K40E.csv", np.array(sample_syn_vec_num_total[i]), delimiter=",")
    np.savetxt(savefol_f + "az_length_per_synapse_K40E.csv", np.array(sample_az_len_total[i]), delimiter=",")
    np.savetxt(savefol_f + "vec_area_per_synapse_K40E.csv", np.array(sample_syn_vec_area_total[i]), delimiter=",")
    np.savetxt(savefol_f + "vec_area_all_K40E.csv", np.array(list(chain.from_iterable(sample_vec_area_total[i]))), delimiter=",")

    # with open("vec_area_all_SD.json", "w", encoding="utf-8") as f:
    #     json.dump(vec_area_total, f, ensure_ascii=False)

    np.savetxt(savefol_f + "vec_NN_per_synapse_K40E.csv", np.array(sample_syn_vec_NN_total[i]), delimiter=",")
    np.savetxt(savefol_f + "vec_NN_all_K40E.csv", np.array(sample_vec_NN_total[i]), delimiter=",")
    np.savetxt(savefol_f + "az_vec_distance_per_synapse_K40E.csv", np.array(sample_syn_az_vec_dists_total[i]), delimiter=",")
    np.savetxt(savefol_f + "az_vec_distance_all_K40E.csv", np.array(sample_az_vec_dists_total[i]), delimiter=",")

E
F
G
H


In [7]:
len(syn_az_vec_dists_ind_li)


59

In [11]:
print(len(az_vec_dists_total))
print(len(vec_NN_total))
print(len(syn_vec_num_total))
len(az_len_total)

33230
33230
586


586

In [10]:
#analyze vesicles and presynapses  get params for WT samples
distance_threshold = 5
vecnum_in_syn = 10
# syn_fol =  "coco_dir_ex_syn"
# vec_fol =  "coco_dir_ex_vec"
# model_syn_name = "syn_az_aug"
# model_vec_name = "vec_aug"
unit = "vecall0"


fname_lists = ["A_0","B_0","C_0","D_0"]

syn_vec_num_total_ctrl = []
# sample_syn_vec_num_total_ctrl = []

az_len_total_ctrl = []
# sample_az_len_total_ctrl = []

vec_area_total_ctrl = []
# sample_vec_area_total_ctrl = []
syn_vec_area_total_ctrl = []
# sample_syn_vec_area_total_ctrl = []

vec_NN_total_ctrl = []
syn_vec_NN_total_ctrl = []

az_vec_dists_total_ctrl = []
syn_az_vec_dists_total_ctrl = []
syn_az_vec_dists_ind_li_ctrl = []
syn_NN_ind_li_ctrl = []


#by sample
# sample_vec_sum_total_ctrl = []
# sample_m_syn_az_vec_dists_total_ctrl = []
# sample_syn_vec_num_cv_total_ctrl= []
# sample_m_syn_az_vec_dists_cv_total_ctrl= []

sample_syn_vec_num_total_ctrl = []
sample_az_len_total_ctrl = []
sample_syn_vec_area_total_ctrl = []
sample_vec_area_total_ctrl = []
# sample_syn_vec_area_total = []

sample_vec_NN_total_ctrl = []
sample_syn_vec_NN_total_ctrl = []
sample_az_vec_dists_total_ctrl = []
sample_syn_az_vec_dists_total_ctrl = []




for m, fname_pre in enumerate(fname_lists):
#     if m>0:
#         continue
    print(fname_pre)
    
    
    save_fol_file = save_fol + "/" +fname_pre +"/"
    
    if os.path.exists(save_fol_file  +"syn_az_len.npy"):
    
        syn_az_len =list(np.load(save_fol_file  +"syn_az_len.npy" , allow_pickle=True))
        syn_vec_num = list(np.load(save_fol_file  +"syn_vec_num.npy", allow_pickle=True))
        syn_vec_area =list(np.load(save_fol_file  +"syn_vec_area.npy", allow_pickle=True))
        syn_vec_NN =list(np.load(save_fol_file  +"syn_vec_NN.npy", allow_pickle=True))
        syn_az_vec_dists =list(np.load(save_fol_file  +"syn_az_vec_dists.npy", allow_pickle=True))
        m_syn_az_vec_dists =list(np.load(save_fol_file  +"m_syn_az_vec_dists.npy", allow_pickle=True))
        m_syn_vec_area = list(np.load(save_fol_file  +"m_syn_vec_area.npy", allow_pickle=True))
        m_syn_vec_NN =list(np.load(save_fol_file  +"m_syn_vec_NN.npy", allow_pickle=True))
        vec_NN =list(np.load(save_fol_file  +"vec_NN.npy", allow_pickle=True))
        az_vec_dists_li =list(np.load(save_fol_file  +"az_vec_dists_li.npy", allow_pickle=True))
        
        syn_num = len(syn_az_len)
        
#         sample_vec_sum_total_ctrl.append(int(np.sum(syn_vec_num)/syn_num))
#         sample_syn_vec_num_cv_total_ctrl.append(np.std(syn_vec_num)/np.mean(syn_vec_num))
#         sample_m_syn_az_vec_dists_total_ctrl.append(np.mean(m_syn_az_vec_dists))
#         sample_m_syn_az_vec_dists_cv_total_ctrl.append(np.std(m_syn_az_vec_dists)/np.mean(m_syn_az_vec_dists))
        
        syn_vec_num_total_ctrl += syn_vec_num
        az_len_total_ctrl += syn_az_len 
        vec_area_total_ctrl += syn_vec_area
        syn_vec_area_total_ctrl += m_syn_vec_area
        vec_NN_total_ctrl += vec_NN
        syn_vec_NN_total_ctrl += m_syn_vec_NN
        az_vec_dists_total_ctrl += az_vec_dists_li 
        syn_az_vec_dists_total_ctrl +=  m_syn_az_vec_dists
        
        
        syn_az_vec_dists_ind_li_ctrl+=syn_az_vec_dists
        syn_NN_ind_li_ctrl+=syn_vec_NN
        
        
        
        #by sample
        sample_syn_vec_num_total_ctrl.append(syn_vec_num)
        sample_az_len_total_ctrl.append(syn_az_len)
        sample_vec_area_total_ctrl.append(syn_vec_area)
        sample_syn_vec_area_total_ctrl.append(m_syn_vec_area)
        sample_vec_NN_total_ctrl.append(vec_NN)
        sample_syn_vec_NN_total_ctrl.append(m_syn_vec_NN)
        sample_az_vec_dists_total_ctrl.append(az_vec_dists_li)
        sample_syn_az_vec_dists_total_ctrl.append(m_syn_az_vec_dists)
        

A_0
B_0
C_0
D_0


In [12]:
save_fol_file

'/home/gpu_data/data8/kinoshita_cfos/EM_analysis//analyzed_snap//D_0/'

In [12]:
#by samples  ctrl
import json
from itertools import chain

# fname_lists =["#10","#20","#30","#40"]

savefol = wsl_pyf + "/WT-vs-K40E/"
os.makedirs(savefol, exist_ok = True)

for i, fname in enumerate(fname_lists):
    
    
    fname = fname[0]
    print(fname)
    savefol_f = wsl_pyf + "/WT-vs-K40E_sample/" + fname +"/"
    os.makedirs(savefol_f, exist_ok = True)
    

    np.savetxt(savefol_f + "vec_num_per_synapse_WT.csv", np.array(sample_syn_vec_num_total_ctrl[i]), delimiter=",")
    np.savetxt(savefol_f + "az_length_per_synapse_WT.csv", np.array(sample_az_len_total_ctrl[i]), delimiter=",")
    np.savetxt(savefol_f + "vec_area_per_synapse_WT.csv", np.array(sample_syn_vec_area_total_ctrl[i]), delimiter=",")
    np.savetxt(savefol_f + "vec_area_all_WT.csv", np.array(list(chain.from_iterable(sample_vec_area_total_ctrl[i]))), delimiter=",")

    # with open("vec_area_all_SD.json", "w", encoding="utf-8") as f:
    #     json.dump(vec_area_total, f, ensure_ascii=False)

    np.savetxt(savefol_f + "vec_NN_per_synapse_WT.csv", np.array(sample_syn_vec_NN_total_ctrl[i]), delimiter=",")
    np.savetxt(savefol_f + "vec_NN_all_WT.csv", np.array(sample_vec_NN_total_ctrl[i]), delimiter=",")
    np.savetxt(savefol_f + "az_vec_distance_per_synapse_WT.csv", np.array(sample_syn_az_vec_dists_total_ctrl[i]), delimiter=",")
    np.savetxt(savefol_f + "az_vec_distance_all_WT.csv", np.array(sample_az_vec_dists_total_ctrl[i]), delimiter=",")

A
B
C
D


In [28]:
len(syn_az_vec_dists[0])

126

In [11]:

from itertools import chain

savefol = wsl_pyf + "//WT-vs-K40E/"
os.makedirs(savefol, exist_ok = True)

np.savetxt(savefol + "vec_num_per_synapse_WT.csv", np.array(syn_vec_num_total_ctrl), delimiter=",")
np.savetxt(savefol + "az_length_per_synapse_WT.csv", np.array(az_len_total_ctrl), delimiter=",")
np.savetxt(savefol + "vec_area_per_synapse_WT.csv", np.array(syn_vec_area_total_ctrl), delimiter=",")
np.savetxt(savefol + "vec_area_all_WT.csv", np.array(list(chain.from_iterable(vec_area_total_ctrl))), delimiter=",")

with open(savefol +"vec_area_per_synapse_all_WT.json", "w", encoding="utf-8") as f:
    json.dump(to_jsonable(vec_area_total_ctrl), f, ensure_ascii=False)

with open(savefol +"vec_NN_per_synapse_all_WT.json", "w", encoding="utf-8") as f:
    json.dump(to_jsonable(syn_NN_ind_li_ctrl), f, ensure_ascii=False)
    
with open(savefol +"az_vec_distance_per_synapse_all_WT.json", "w", encoding="utf-8") as f:
    json.dump(to_jsonable(syn_az_vec_dists_ind_li_ctrl), f, ensure_ascii=False)
# with open("vec_area_all_SD.json", "w", encoding="utf-8") as f:
#     json.dump(vec_area_total, f, ensure_ascii=False)
    
np.savetxt(savefol + "vec_NN_per_synapse_WT.csv", np.array(syn_vec_NN_total_ctrl), delimiter=",")
np.savetxt(savefol + "vec_NN_all_WT.csv", np.array(vec_NN_total_ctrl), delimiter=",")
np.savetxt(savefol + "az_vec_distance_per_synapse_WT.csv", np.array(syn_az_vec_dists_total_ctrl), delimiter=",")
np.savetxt(savefol + "az_vec_distance_all_WT.csv", np.array(az_vec_dists_total_ctrl), delimiter=",")